ConvNeXt-Tiny experiment V3

ConvNeXt-Tiny with higher-resolution inputs plus cosine LR schedule and longer warmup, tuned from V2.


1. Libary and device setup

In [ ]:
import copy
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms, models
from collections import Counter

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

2. Class count and Disease name

In [ ]:
COMPETITION_DATA_DIR = Path("/kaggle/input/competitions/cassava-leaf-disease-classification")
WEIGHTS_PATH = Path("/kaggle/input/datasets/lawhan/convnext-pretrained-weights/convnext_tiny-983f1562.pth")  # Update this manually to match your attached Kaggle input

TRAIN_CSV_PATH = COMPETITION_DATA_DIR / "train.csv"
LABEL_MAP_PATH = COMPETITION_DATA_DIR / "label_num_to_disease_map.json"
TRAIN_IMAGE_DIR = COMPETITION_DATA_DIR / "train_images"
TEST_CSV_PATH = COMPETITION_DATA_DIR / "sample_submission.csv"
TEST_IMAGE_DIR = COMPETITION_DATA_DIR / "test_images"

with open(LABEL_MAP_PATH, "r") as f:
    label_num_to_disease_map = json.load(f)

train_df = pd.read_csv(TRAIN_CSV_PATH)
print(train_df.head())


3. Train and Split Validation and Distribution

In [ ]:
# Split dataset (80% train, 20% validation)
train_df_split, valid_df_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=42
)

# Count labels
train_counts = Counter(train_df_split["label"])
valid_counts = Counter(valid_df_split["label"])

print("Training set class counts:\n")
for class_idx, count in sorted(train_counts.items()):
    print(f"{label_num_to_disease_map[str(class_idx)]}: {count}")

print("\nValidation set class counts:\n")
for class_idx, count in sorted(valid_counts.items()):
    print(f"{label_num_to_disease_map[str(class_idx)]}: {count}")



4. ConvNeXt-Tiny setup (offline Kaggle input weights)


In [ ]:
from torchvision import models

def resolve_weights_path(weights_path=WEIGHTS_PATH):
    weights_path = Path(weights_path)

    if weights_path.is_dir():
        candidate_patterns = [
            "convnext_tiny-983f1562.pth",
            "convnext_tiny*.pth",
            "*.pth",
        ]
        for pattern in candidate_patterns:
            candidates = sorted(weights_path.rglob(pattern))
            if candidates:
                resolved_path = candidates[0]
                print(f"Resolved checkpoint inside directory: {resolved_path}")
                return resolved_path
        raise FileNotFoundError(
            f"No ConvNeXt-Tiny checkpoint was found under directory: {weights_path}"
        )

    if not weights_path.exists():
        raise FileNotFoundError(f"Offline checkpoint not found: {weights_path}")

    return weights_path


def build_model(num_classes=5, weights_path=WEIGHTS_PATH, dropout_p=0.3):
    resolved_weights_path = resolve_weights_path(weights_path)

    model = models.convnext_tiny(weights=None)
    state_dict = torch.load(resolved_weights_path, map_location="cpu")
    model.load_state_dict(state_dict)

    feature_dim = model.classifier[-1].in_features
    model.classifier = nn.Sequential(
        model.classifier[0],
        model.classifier[1],
        nn.Dropout(p=dropout_p),
        nn.Linear(feature_dim, num_classes),
    )
    return model


In [ ]:
learning_rate = 2e-4
weight_decay = 1e-4
dropout_p = 0.3
label_smoothing = 0.1
batch_size = 16
image_resize = 288
image_crop = 256
warmup_epochs = 3

model = build_model(dropout_p=dropout_p).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(1, 10 - warmup_epochs),
)


5. Creating Dataset Class

In [ ]:
class CassavaDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image_id"]
        label = self.df.iloc[idx]["label"]

        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

6. Train and evaluation transforms (ConvNeXt-Tiny higher-resolution preprocessing)


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((image_resize, image_resize)),
    transforms.RandomCrop((image_crop, image_crop)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((image_resize, image_resize)),
    transforms.CenterCrop((image_crop, image_crop)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


7. Creating Train and Valid dataset

In [ ]:
train_dataset = CassavaDataset(
    df=train_df_split,
    image_dir=TRAIN_IMAGE_DIR,
    transform=train_transform
)

valid_dataset = CassavaDataset(
    df=valid_df_split,
    image_dir=TRAIN_IMAGE_DIR,
    transform=eval_transform
)


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)


8. Training ConvNeXt-Tiny model


In [ ]:
num_epochs = 10
early_stop_patience = 4
epochs_without_improvement = 0
history = []
best_val_accuracy = 0.0
best_model_state = copy.deepcopy(model.state_dict())
best_epoch = 0
best_checkpoint_path = Path("best_convnext_tiny_v3_best.pth")

for epoch in range(num_epochs):
    # Linear warmup for the first few epochs, then cosine decay.
    if epoch < warmup_epochs:
        warmup_lr = learning_rate * (epoch + 1) / warmup_epochs
        for group in optimizer.param_groups:
            group["lr"] = warmup_lr
    elif epoch == warmup_epochs:
        for group in optimizer.param_groups:
            group["lr"] = learning_rate

    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_accuracy = 100 * train_correct / train_total
    avg_train_loss = running_loss / train_total

    model.eval()
    valid_running_loss = 0.0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            _, predicted = torch.max(outputs, 1)

            valid_running_loss += loss.item() * labels.size(0)
            valid_total += labels.size(0)
            valid_correct += (predicted == labels).sum().item()

    valid_accuracy = 100 * valid_correct / valid_total
    avg_valid_loss = valid_running_loss / valid_total

    history.append(
        {
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "train_accuracy": train_accuracy,
            "valid_loss": avg_valid_loss,
            "valid_accuracy": valid_accuracy,
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
    )

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Training Loss: {avg_train_loss:.4f}")
    print(f"Training Accuracy: {train_accuracy:.2f}%")
    print(f"Validation Loss: {avg_valid_loss:.4f}")
    print(f"Validation Accuracy: {valid_accuracy:.2f}%")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}\n")

    if valid_accuracy > best_val_accuracy:
        best_val_accuracy = valid_accuracy
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch + 1
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch >= warmup_epochs:
        scheduler.step()

    if epochs_without_improvement >= early_stop_patience:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

history_df = pd.DataFrame(history)
model.load_state_dict(best_model_state)
torch.save(best_model_state, best_checkpoint_path)

print(f"Best validation accuracy: {best_val_accuracy:.2f}%")
print(f"Best epoch: {best_epoch}")
print(f"Saved {best_checkpoint_path}")
history_df


8b. Validation metrics and visualizations

These cells turn the best checkpoint into presentation-ready evaluation plots so you can compare learning curves,
per-class performance, and the model's main failure modes.


In [ ]:
label_lookup = {int(k): v for k, v in label_num_to_disease_map.items()}
label_order = sorted(valid_df_split["label"].unique())
class_names = [label_lookup[label] for label in label_order]

valid_targets = []
valid_predictions = []
valid_confidences = []

model.eval()
with torch.no_grad():
    for images, labels in valid_loader:
        images = images.to(device)
        outputs = model(images)
        probabilities = torch.softmax(outputs, dim=1)
        confidences, predicted = probabilities.max(dim=1)

        valid_targets.extend(labels.cpu().numpy().tolist())
        valid_predictions.extend(predicted.cpu().numpy().tolist())
        valid_confidences.extend(confidences.cpu().numpy().tolist())

valid_results_df = valid_dataset.df.reset_index(drop=True).copy()
valid_results_df["pred_label"] = valid_predictions
valid_results_df["true_name"] = valid_results_df["label"].map(label_lookup)
valid_results_df["pred_name"] = valid_results_df["pred_label"].map(label_lookup)
valid_results_df["confidence"] = valid_confidences
valid_results_df["correct"] = valid_results_df["label"] == valid_results_df["pred_label"]

print(f"Validation accuracy from best checkpoint: {100 * valid_results_df['correct'].mean():.2f}%")
print(f"Best epoch from training: {best_epoch}")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(history_df["epoch"], history_df["train_accuracy"], marker="o", linewidth=2, label="Train accuracy")
axes[0].plot(history_df["epoch"], history_df["valid_accuracy"], marker="o", linewidth=2, label="Validation accuracy")
axes[0].axvline(best_epoch, color="crimson", linestyle="--", alpha=0.8, label=f"Best epoch ({best_epoch})")
axes[0].set_title("ConvNeXt-Tiny accuracy over epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy (%)")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_loss"], marker="o", linewidth=2, label="Train loss")
axes[1].plot(history_df["epoch"], history_df["valid_loss"], marker="o", linewidth=2, label="Validation loss")
axes[1].axvline(best_epoch, color="crimson", linestyle="--", alpha=0.8, label=f"Best epoch ({best_epoch})")
axes[1].set_title("ConvNeXt-Tiny loss over epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].grid(alpha=0.3)
axes[1].legend()

fig.tight_layout()
plt.show()

conf_mat = confusion_matrix(valid_targets, valid_predictions, labels=label_order)
conf_mat_df = pd.DataFrame(conf_mat, index=class_names, columns=class_names)
conf_mat_pct = conf_mat_df.div(conf_mat_df.sum(axis=1).replace(0, 1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
count_im = axes[0].imshow(conf_mat_df.values, cmap="Blues")
axes[0].set_title("ConvNeXt-Tiny validation confusion matrix (counts)")
axes[0].set_xticks(range(len(class_names)))
axes[0].set_xticklabels(class_names, rotation=45, ha="right")
axes[0].set_yticks(range(len(class_names)))
axes[0].set_yticklabels(class_names)
axes[0].set_xlabel("Predicted label")
axes[0].set_ylabel("True label")
plt.colorbar(count_im, ax=axes[0], fraction=0.046, pad=0.04)

for row_idx in range(conf_mat_df.shape[0]):
    for col_idx in range(conf_mat_df.shape[1]):
        value = conf_mat_df.iat[row_idx, col_idx]
        text_color = "white" if value > conf_mat_df.values.max() * 0.5 else "black"
        axes[0].text(col_idx, row_idx, int(value), ha="center", va="center", color=text_color, fontsize=10)

pct_im = axes[1].imshow(conf_mat_pct.values, cmap="Greens", vmin=0, vmax=100)
axes[1].set_title("ConvNeXt-Tiny validation confusion matrix (row %)")
axes[1].set_xticks(range(len(class_names)))
axes[1].set_xticklabels(class_names, rotation=45, ha="right")
axes[1].set_yticks(range(len(class_names)))
axes[1].set_yticklabels(class_names)
axes[1].set_xlabel("Predicted label")
axes[1].set_ylabel("True label")
plt.colorbar(pct_im, ax=axes[1], fraction=0.046, pad=0.04, label="Row percentage")

for row_idx in range(conf_mat_pct.shape[0]):
    for col_idx in range(conf_mat_pct.shape[1]):
        value = conf_mat_pct.iat[row_idx, col_idx]
        text_color = "white" if value > 50 else "black"
        axes[1].text(col_idx, row_idx, f"{value:.1f}%", ha="center", va="center", color=text_color, fontsize=10)

fig.tight_layout()
plt.show()

report_df = pd.DataFrame(
    classification_report(
        valid_targets,
        valid_predictions,
        labels=label_order,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
).transpose()

per_class_report_df = report_df.loc[class_names, ["precision", "recall", "f1-score", "support"]].copy()

print(f"Validation accuracy: {100 * valid_results_df['correct'].mean():.2f}%")
print(f"Macro precision: {report_df.loc['macro avg', 'precision']:.3f}")
print(f"Macro recall: {report_df.loc['macro avg', 'recall']:.3f}")
print(f"Macro F1 score: {report_df.loc['macro avg', 'f1-score']:.3f}")
print(f"Weighted F1 score: {report_df.loc['weighted avg', 'f1-score']:.3f}")
display(per_class_report_df.round(3))

ax = per_class_report_df[["precision", "recall", "f1-score"]].plot(
    kind="bar",
    figsize=(12, 5),
    ylim=(0, 1),
    rot=30,
)
ax.set_title("ConvNeXt-Tiny per-class precision, recall, and F1")
ax.set_ylabel("Score")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

confusion_pairs = []
for row_idx, true_name in enumerate(class_names):
    for col_idx, pred_name in enumerate(class_names):
        count = int(conf_mat[row_idx, col_idx])
        if row_idx != col_idx and count > 0:
            confusion_pairs.append(
                {
                    "true_label": true_name,
                    "predicted_label": pred_name,
                    "count": count,
                }
            )

if confusion_pairs:
    most_confused_df = pd.DataFrame(confusion_pairs).sort_values("count", ascending=False).head(10).reset_index(drop=True)
    print("Most common validation mistakes:")
    display(most_confused_df)
else:
    print("No validation misclassifications were found.")


In [ ]:
def show_prediction_examples(results_df, title, only_correct, n=6):
    subset = results_df[results_df["correct"] == only_correct].copy()
    if subset.empty:
        print(f"No {'correct' if only_correct else 'incorrect'} predictions to display.")
        return

    subset = subset.sort_values("confidence", ascending=False).head(n).reset_index(drop=True)
    ncols = 3
    nrows = int(np.ceil(len(subset) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4.5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, row in zip(axes, subset.to_dict("records")):
        image = Image.open(TRAIN_IMAGE_DIR / row["image_id"]).convert("RGB")
        ax.imshow(image)
        ax.axis("off")
        title_color = "forestgreen" if row["correct"] else "crimson"
        ax.set_title(
            f"{row['image_id']}\nTrue: {row['true_name']}\nPred: {row['pred_name']} ({row['confidence']:.1%})",
            color=title_color,
            fontsize=10,
        )

    for ax in axes[len(subset):]:
        ax.axis("off")

    fig.suptitle(title, fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


show_prediction_examples(
    valid_results_df,
    "ConvNeXt-Tiny validation examples: most confident correct predictions",
    only_correct=True,
    n=6,
)
show_prediction_examples(
    valid_results_df,
    "ConvNeXt-Tiny validation examples: most confident mistakes",
    only_correct=False,
    n=6,
)


8c. Save evaluation artifacts

This optional cell saves the evaluation figures and tables to disk so you can reuse them in reports,
slides, or later comparisons without rerunning the notebook outputs manually.


In [ ]:
artifact_dir = Path("evaluation_artifacts") / "convnext_tiny_v3"
artifact_dir.mkdir(parents=True, exist_ok=True)

summary_metrics = {
    "model_name": "ConvNeXt-Tiny",
    "best_epoch": int(best_epoch),
    "best_val_accuracy": float(best_val_accuracy),
    "validation_accuracy_from_predictions": float(100 * valid_results_df["correct"].mean()),
    "macro_precision": float(report_df.loc["macro avg", "precision"]),
    "macro_recall": float(report_df.loc["macro avg", "recall"]),
    "macro_f1": float(report_df.loc["macro avg", "f1-score"]),
    "weighted_f1": float(report_df.loc["weighted avg", "f1-score"]),
    "num_validation_samples": int(len(valid_results_df)),
    "num_correct_predictions": int(valid_results_df["correct"].sum()),
    "num_incorrect_predictions": int((~valid_results_df["correct"]).sum()),
}

with open(artifact_dir / "summary_metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary_metrics, f, indent=2)

history_df.to_csv(artifact_dir / "training_history.csv", index=False)
valid_results_df.to_csv(artifact_dir / "validation_predictions.csv", index=False)
report_df.to_csv(artifact_dir / "classification_report.csv")
per_class_report_df.to_csv(artifact_dir / "per_class_report.csv")
conf_mat_df.to_csv(artifact_dir / "confusion_matrix_counts.csv")
conf_mat_pct.to_csv(artifact_dir / "confusion_matrix_row_percent.csv")

if "most_confused_df" in globals():
    most_confused_df.to_csv(artifact_dir / "most_confused_pairs.csv", index=False)

curve_fig, curve_axes = plt.subplots(1, 2, figsize=(16, 5))

curve_axes[0].plot(history_df["epoch"], history_df["train_accuracy"], marker="o", linewidth=2, label="Train accuracy")
curve_axes[0].plot(history_df["epoch"], history_df["valid_accuracy"], marker="o", linewidth=2, label="Validation accuracy")
curve_axes[0].axvline(best_epoch, color="crimson", linestyle="--", alpha=0.8, label=f"Best epoch ({best_epoch})")
curve_axes[0].set_title("ConvNeXt-Tiny accuracy over epochs")
curve_axes[0].set_xlabel("Epoch")
curve_axes[0].set_ylabel("Accuracy (%)")
curve_axes[0].grid(alpha=0.3)
curve_axes[0].legend()

curve_axes[1].plot(history_df["epoch"], history_df["train_loss"], marker="o", linewidth=2, label="Train loss")
curve_axes[1].plot(history_df["epoch"], history_df["valid_loss"], marker="o", linewidth=2, label="Validation loss")
curve_axes[1].axvline(best_epoch, color="crimson", linestyle="--", alpha=0.8, label=f"Best epoch ({best_epoch})")
curve_axes[1].set_title("ConvNeXt-Tiny loss over epochs")
curve_axes[1].set_xlabel("Epoch")
curve_axes[1].set_ylabel("Loss")
curve_axes[1].grid(alpha=0.3)
curve_axes[1].legend()

curve_fig.tight_layout()
curve_fig.savefig(artifact_dir / "accuracy_loss_curves.png", dpi=200, bbox_inches="tight")
plt.close(curve_fig)

conf_fig, conf_axes = plt.subplots(1, 2, figsize=(20, 7))
count_im = conf_axes[0].imshow(conf_mat_df.values, cmap="Blues")
conf_axes[0].set_title("ConvNeXt-Tiny validation confusion matrix (counts)")
conf_axes[0].set_xticks(range(len(class_names)))
conf_axes[0].set_xticklabels(class_names, rotation=45, ha="right")
conf_axes[0].set_yticks(range(len(class_names)))
conf_axes[0].set_yticklabels(class_names)
conf_axes[0].set_xlabel("Predicted label")
conf_axes[0].set_ylabel("True label")
plt.colorbar(count_im, ax=conf_axes[0], fraction=0.046, pad=0.04)

for row_idx in range(conf_mat_df.shape[0]):
    for col_idx in range(conf_mat_df.shape[1]):
        value = conf_mat_df.iat[row_idx, col_idx]
        text_color = "white" if value > conf_mat_df.values.max() * 0.5 else "black"
        conf_axes[0].text(col_idx, row_idx, int(value), ha="center", va="center", color=text_color, fontsize=10)

pct_im = conf_axes[1].imshow(conf_mat_pct.values, cmap="Greens", vmin=0, vmax=100)
conf_axes[1].set_title("ConvNeXt-Tiny validation confusion matrix (row %)")
conf_axes[1].set_xticks(range(len(class_names)))
conf_axes[1].set_xticklabels(class_names, rotation=45, ha="right")
conf_axes[1].set_yticks(range(len(class_names)))
conf_axes[1].set_yticklabels(class_names)
conf_axes[1].set_xlabel("Predicted label")
conf_axes[1].set_ylabel("True label")
plt.colorbar(pct_im, ax=conf_axes[1], fraction=0.046, pad=0.04, label="Row percentage")

for row_idx in range(conf_mat_pct.shape[0]):
    for col_idx in range(conf_mat_pct.shape[1]):
        value = conf_mat_pct.iat[row_idx, col_idx]
        text_color = "white" if value > 50 else "black"
        conf_axes[1].text(col_idx, row_idx, f"{value:.1f}%", ha="center", va="center", color=text_color, fontsize=10)

conf_fig.tight_layout()
conf_fig.savefig(artifact_dir / "confusion_matrices.png", dpi=200, bbox_inches="tight")
plt.close(conf_fig)

metrics_ax = per_class_report_df[["precision", "recall", "f1-score"]].plot(
    kind="bar",
    figsize=(12, 5),
    ylim=(0, 1),
    rot=30,
)
metrics_ax.set_title("ConvNeXt-Tiny per-class precision, recall, and F1")
metrics_ax.set_ylabel("Score")
metrics_ax.grid(axis="y", alpha=0.3)
metrics_fig = metrics_ax.get_figure()
metrics_fig.tight_layout()
metrics_fig.savefig(artifact_dir / "per_class_metrics.png", dpi=200, bbox_inches="tight")
plt.close(metrics_fig)


def save_prediction_examples(results_df, title, only_correct, filename, n=6):
    subset = results_df[results_df["correct"] == only_correct].copy()
    if subset.empty:
        return False

    subset = subset.sort_values("confidence", ascending=False).head(n).reset_index(drop=True)
    ncols = 3
    nrows = int(np.ceil(len(subset) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4.5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, row in zip(axes, subset.to_dict("records")):
        image = Image.open(TRAIN_IMAGE_DIR / row["image_id"]).convert("RGB")
        ax.imshow(image)
        ax.axis("off")
        title_color = "forestgreen" if row["correct"] else "crimson"
        ax.set_title(
            f"{row['image_id']}\nTrue: {row['true_name']}\nPred: {row['pred_name']} ({row['confidence']:.1%})",
            color=title_color,
            fontsize=10,
        )

    for ax in axes[len(subset):]:
        ax.axis("off")

    fig.suptitle(title, fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(artifact_dir / filename, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return True


saved_correct_examples = save_prediction_examples(
    valid_results_df,
    "ConvNeXt-Tiny validation examples: most confident correct predictions",
    only_correct=True,
    filename="examples_correct.png",
    n=6,
)
saved_incorrect_examples = save_prediction_examples(
    valid_results_df,
    "ConvNeXt-Tiny validation examples: most confident mistakes",
    only_correct=False,
    filename="examples_mistakes.png",
    n=6,
)

saved_artifacts = sorted(path.name for path in artifact_dir.iterdir())
print(f"Saved evaluation artifacts to {artifact_dir.resolve()}")
print(f"Saved correct example grid: {saved_correct_examples}")
print(f"Saved mistake example grid: {saved_incorrect_examples}")
saved_artifacts


8d. Export best model bundle to `.pt`

This cell saves the best ConvNeXt-Tiny checkpoint plus lightweight metadata into a reusable `.pt` file.


In [ ]:
best_checkpoint_path = Path("best_convnext_tiny_v3_best.pth")
best_model_pt_path = Path("best_convnext_tiny_v3_model.pt")

if "best_model_state" in globals():
    export_state_dict = best_model_state
else:
    export_state_dict = torch.load(best_checkpoint_path, map_location="cpu")

model_pt_bundle = {
    "model_name": "convnext_tiny",
    "num_classes": 5,
    
    "image_resize": image_resize,
    "image_crop": image_crop,
    "label_smoothing": label_smoothing,
    "dropout_p": dropout_p,
    "optimizer": "AdamW",
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "batch_size": batch_size,
    "warmup_epochs": warmup_epochs,

    "best_epoch": best_epoch if "best_epoch" in globals() else None,
    "best_val_accuracy": best_val_accuracy if "best_val_accuracy" in globals() else None,
    "label_map": label_lookup if "label_lookup" in globals() else None,
    "class_names": class_names if "class_names" in globals() else None,
    "history": history_df.to_dict(orient="records") if "history_df" in globals() else None,
    "summary_metrics": summary_metrics if "summary_metrics" in globals() else None,
    "artifact_dir": str(artifact_dir) if "artifact_dir" in globals() else None,
    "state_dict": export_state_dict,
}

torch.save(model_pt_bundle, best_model_pt_path)
print(f"Saved {best_model_pt_path.resolve()}")
print(f"Bundle keys: {list(model_pt_bundle.keys())}")


8e. Reload the exported `.pt` bundle

Use this optional cell to confirm the saved file and inspect its metadata without rebuilding the model first.


In [ ]:
loaded_model_pt_bundle = torch.load(best_model_pt_path, map_location="cpu")
loaded_model_summary = {
    key: value for key, value in loaded_model_pt_bundle.items() if key != "state_dict"
}
loaded_model_summary


In [ ]:
# Test set inference using the best validation checkpoint
test_df = pd.read_csv(TEST_CSV_PATH)
test_image_dir = TEST_IMAGE_DIR


class CassavaTestDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image_id"]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, img_name


test_dataset = CassavaTestDataset(test_df, test_image_dir, transform=eval_transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model.eval()
preds = []
ids = []
with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        preds.extend(predicted.cpu().numpy().tolist())
        ids.extend(image_ids)

submission = pd.DataFrame({"image_id": ids, "label": preds})
submission_path = Path("submission.csv")
submission.to_csv(submission_path, index=False)
print(f"Saved {submission_path}")
